In [2]:
import os
import gc
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")
os.environ["PYTHONHASHSEED"] = "42"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

2. Configuracion general

In [3]:
@dataclass
class Config:
    input_file: str = "transformed_it_ot.csv"
    label_col: str = "traffic_label"
    attack_col: str = "attack_type"
    leakage_cols: tuple = ("attack_type", "line_number")
    zero_day_attack: int = 5
    test_normal_frac: float = 0.20
    random_state: int = 42

    k_best: int = 10
    pca_variance: float = 0.90
    use_pca: bool = True 

    smote_strategy: float = "auto"
    smote_k_neighbors: int = 5

    n_splits: int = 3
    dnn_epochs: int = 20
    dnn_batch_size: int = 1024
    dnn_patience: int = 5

cfg = Config()
print(cfg)

Config(input_file='transformed_it_ot.csv', label_col='traffic_label', attack_col='attack_type', leakage_cols=('attack_type', 'line_number'), zero_day_attack=5, test_normal_frac=0.2, random_state=42, k_best=10, pca_variance=0.9, use_pca=True, smote_strategy='auto', smote_k_neighbors=5, n_splits=3, dnn_epochs=20, dnn_batch_size=1024, dnn_patience=5)


3. Carga del dataset

In [4]:
print("Cargando dataset...")
df = pd.read_csv(cfg.input_file, low_memory=False, encoding="utf-8-sig")

df.columns = (
    df.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
    .str.lower()
)

print("Columnas detectadas:")
print(df.columns.tolist())

required_cols = [cfg.label_col, cfg.attack_col]
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"No existe la columna requerida: {col}")

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[cfg.label_col, cfg.attack_col]).reset_index(drop=True)

df[cfg.label_col] = pd.to_numeric(df[cfg.label_col], errors="coerce").astype("int32")
df[cfg.attack_col] = pd.to_numeric(df[cfg.attack_col], errors="coerce").astype("int32")

print("Forma inicial:", df.shape)
gc.collect()

Cargando dataset...
Columnas detectadas:
['src_port', 'dst_port', 'protocol', 'duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_packets', 'dst2src_packets', 'src2dst_bytes', 'dst2src_bytes', 'src2dst_duration_ms', 'dst2src_duration_ms', 'application_name', 'application_category', 'attack_type', 'traffic_label', 'line_number', 'log_length', 'device_type', 'hour', 'minute', 'second']
Forma inicial: (3907536, 22)


0

4. Zero-Day split

In [5]:
# =========================================================
# ZERO-DAY SPLIT + REETIQUETADO SUPERVISADO
# attack_type == 0 -> normal
# attack_type != 0 -> ataque
# attack_type == 5 -> zero-day, solo en test
# =========================================================

attack_series = df[cfg.attack_col]

# Crear nueva etiqueta binaria supervisada desde attack_type
# 0 = normal, 1 = ataque
df["target_label"] = (attack_series != 0).astype("int32")

# Separar conjuntos
zero_day_df = df[attack_series == cfg.zero_day_attack].copy()
normal_df = df[attack_series == 0].copy()

normal_test_df = normal_df.sample(
    frac=cfg.test_normal_frac,
    random_state=cfg.random_state
).copy()

normal_train_df = normal_df.drop(index=normal_test_df.index).copy()

known_attacks_df = df[
    (attack_series != 0) &
    (attack_series != cfg.zero_day_attack)
].copy()

train_df = pd.concat(
    [normal_train_df, known_attacks_df],
    axis=0,
    ignore_index=True
).sample(frac=1.0, random_state=cfg.random_state).reset_index(drop=True)

test_df = pd.concat(
    [zero_day_df, normal_test_df],
    axis=0,
    ignore_index=True
).sample(frac=1.0, random_state=cfg.random_state).reset_index(drop=True)

print("Train total:", train_df.shape)
print("Test total :", test_df.shape)

print("Distribucion train target_label:")
print(train_df["target_label"].value_counts())

print("Distribucion test target_label:")
print(test_df["target_label"].value_counts())

print("Zero-day presente solo en test:")
print("Train attack_type 5:", (train_df[cfg.attack_col] == cfg.zero_day_attack).sum())
print("Test  attack_type 5:", (test_df[cfg.attack_col] == cfg.zero_day_attack).sum())

gc.collect()

Train total: (3726899, 23)
Test total : (180637, 23)
Distribucion train target_label:
target_label
1    3346189
0     380710
Name: count, dtype: int64
Distribucion test target_label:
target_label
0    95178
1    85459
Name: count, dtype: int64
Zero-day presente solo en test:
Train attack_type 5: 0
Test  attack_type 5: 85459


0

5. Eliminacion de leakage y separacion X/y

In [6]:
# =========================================================
# ELIMINACION DE LEAKAGE DESPUES DEL SPLIT
# Se eliminan attack_type y line_number de ambos conjuntos
# =========================================================

train_df = train_df.drop(columns=list(cfg.leakage_cols), errors="ignore")
test_df = test_df.drop(columns=list(cfg.leakage_cols), errors="ignore")

if "target_label" not in train_df.columns or "target_label" not in test_df.columns:
    raise KeyError("No existe target_label despues del split.")

y_train = train_df["target_label"].astype("int32").values
X_train = train_df.drop(columns=["target_label"])

y_test = test_df["target_label"].astype("int32").values
X_test = test_df.drop(columns=["target_label"])

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

del train_df, test_df, normal_df, normal_test_df, normal_train_df, known_attacks_df, zero_day_df
gc.collect()

X_train: (3726899, 20)
X_test : (180637, 20)


0

6. Reduccion de memoria

In [7]:
def downcast_numeric(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    for col in df_out.columns:
        if pd.api.types.is_float_dtype(df_out[col]):
            df_out[col] = pd.to_numeric(df_out[col], downcast="float")
        elif pd.api.types.is_integer_dtype(df_out[col]):
            df_out[col] = pd.to_numeric(df_out[col], downcast="integer")
    return df_out

X_train = downcast_numeric(X_train)
X_test = downcast_numeric(X_test)

print("Memoria train (MB):", round(X_train.memory_usage(deep=True).sum() / 1024**2, 2))
print("Memoria test  (MB):", round(X_test.memory_usage(deep=True).sum() / 1024**2, 2))

gc.collect()

Memoria train (MB): 213.25
Memoria test  (MB): 10.34


0

7. StandardScaler

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train.values.astype("float32"))
X_test_scaled = scaler.transform(X_test.values.astype("float32"))

X_train_scaled = X_train_scaled.astype("float32")
X_test_scaled = X_test_scaled.astype("float32")

print("StandardScaler completado.")
gc.collect()

StandardScaler completado.


0

8. SelectKBest

In [9]:
n_features = X_train_scaled.shape[1]
k_best = min(cfg.k_best, n_features)

selector = SelectKBest(score_func=mutual_info_classif, k=k_best)
X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest = selector.transform(X_test_scaled)

X_train_kbest = X_train_kbest.astype("float32")
X_test_kbest = X_test_kbest.astype("float32")

selected_idx = selector.get_support(indices=True)
print("Indices seleccionados:", selected_idx)
print("Shape train kbest:", X_train_kbest.shape)
print("Shape test kbest :", X_test_kbest.shape)

gc.collect()

Indices seleccionados: [ 3  5  8  9 10 11 12 13 16 17]
Shape train kbest: (3726899, 10)
Shape test kbest : (180637, 10)


299

9. PCA opcional

In [10]:
if cfg.use_pca:
    pca = PCA(
        n_components=cfg.pca_variance,
        svd_solver="full",
        random_state=cfg.random_state
    )

    X_train_final = pca.fit_transform(X_train_kbest)
    X_test_final = pca.transform(X_test_kbest)

    X_train_final = X_train_final.astype("float32")
    X_test_final = X_test_final.astype("float32")

    print("PCA activado")
    print("Componentes PCA:", pca.n_components_)
    print("Varianza acumulada:", round(pca.explained_variance_ratio_.sum(), 4))
    print("Shape train final:", X_train_final.shape)
    print("Shape test final :", X_test_final.shape)
else:
    X_train_final = X_train_kbest
    X_test_final = X_test_kbest
    print("PCA desactivado")
    print("Shape train final:", X_train_final.shape)
    print("Shape test final :", X_test_final.shape)

gc.collect()

PCA activado
Componentes PCA: 6
Varianza acumulada: 0.9313
Shape train final: (3726899, 6)
Shape test final : (180637, 6)


0

10. SMOTE solo sobre train

In [11]:
print("Distribucion antes de SMOTE:")
print(pd.Series(y_train).value_counts())

clases = np.unique(y_train)

if len(clases) < 2:
    raise ValueError("y_train tiene una sola clase. Revisa el split.")

minority_count = int(np.min(np.bincount(y_train)))
k_neighbors = min(
    cfg.smote_k_neighbors,
    max(1, minority_count - 1)
)

smote = SMOTE(
    sampling_strategy=cfg.smote_strategy,
    random_state=cfg.random_state,
    k_neighbors=k_neighbors
)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_final,
    y_train
)

X_train_bal = X_train_bal.astype("float32")
y_train_bal = y_train_bal.astype("int32")

print("Distribucion despues:")
print(pd.Series(y_train_bal).value_counts())

gc.collect()

Distribucion antes de SMOTE:
1    3346189
0     380710
Name: count, dtype: int64
Distribucion despues:
1    3346189
0    3346189
Name: count, dtype: int64


23

11. Modelos base

In [12]:
def build_xgb(y_ref):
    pos = max(1, int(np.sum(y_ref == 1)))
    neg = max(1, int(np.sum(y_ref == 0)))

    return XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=180,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=1,
        subsample=0.85,
        colsample_bytree=0.85,
        tree_method="hist",
        n_jobs=-1,
        random_state=cfg.random_state,
        scale_pos_weight=neg / pos
    )

def build_lgbm(y_ref):
    pos = max(1, int(np.sum(y_ref == 1)))
    neg = max(1, int(np.sum(y_ref == 0)))

    return LGBMClassifier(
        objective="binary",
        n_estimators=220,
        learning_rate=0.04,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        n_jobs=-1,
        random_state=cfg.random_state,
        scale_pos_weight=neg / pos,
        verbosity=-1
    )

def build_dnn(input_dim: int) -> tf.keras.Model:
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.20),
        Dense(32, activation="relu"),
        BatchNormalization(),
        Dropout(0.15),
        Dense(16, activation="relu"),
        Dropout(0.10),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

12. OOF para meta-learner

In [13]:
def fit_predict_fold(X_tr, y_tr, X_va, y_va):
    xgb = build_xgb(y_tr)
    lgbm = build_lgbm(y_tr)
    dnn = build_dnn(X_tr.shape[1])

    xgb.fit(X_tr, y_tr)
    lgbm.fit(X_tr, y_tr)

    es = EarlyStopping(monitor="val_loss", patience=cfg.dnn_patience, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=max(2, cfg.dnn_patience // 2),
        min_lr=1e-6
    )

    dnn.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=cfg.dnn_epochs,
        batch_size=cfg.dnn_batch_size,
        verbose=0,
        callbacks=[es, rlrop]
    )

    p_xgb = xgb.predict_proba(X_va)[:, 1]
    p_lgbm = lgbm.predict_proba(X_va)[:, 1]
    p_dnn = dnn.predict(X_va, verbose=0).reshape(-1)

    return np.column_stack([p_xgb, p_lgbm, p_dnn]).astype("float32")


skf = StratifiedKFold(
    n_splits=cfg.n_splits,
    shuffle=True,
    random_state=cfg.random_state
)

oof_meta = np.zeros((X_train_bal.shape[0], 3), dtype="float32")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_bal, y_train_bal), start=1):
    X_tr, X_va = X_train_bal[tr_idx], X_train_bal[va_idx]
    y_tr, y_va = y_train_bal[tr_idx], y_train_bal[va_idx]

    oof_meta[va_idx] = fit_predict_fold(X_tr, y_tr, X_va, y_va)
    print(f"Fold {fold} completado")

meta_learner = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="lbfgs"
)

meta_learner.fit(oof_meta, y_train_bal)
print("Meta-learner entrenado.")

gc.collect()

Fold 1 completado
Fold 2 completado
Fold 3 completado
Meta-learner entrenado.


6848

13. Entrenamiento final

In [14]:
final_xgb = build_xgb(y_train_bal)
final_lgbm = build_lgbm(y_train_bal)
final_dnn = build_dnn(X_train_bal.shape[1])

final_xgb.fit(X_train_bal, y_train_bal)
final_lgbm.fit(X_train_bal, y_train_bal)

es = EarlyStopping(monitor="val_loss", patience=cfg.dnn_patience, restore_best_weights=True)
rlrop = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=max(2, cfg.dnn_patience // 2),
    min_lr=1e-6
)

final_dnn.fit(
    X_train_bal, y_train_bal,
    validation_split=0.15,
    epochs=cfg.dnn_epochs,
    batch_size=cfg.dnn_batch_size,
    verbose=0,
    callbacks=[es, rlrop]
)

print("Modelos base finales entrenados.")
gc.collect()

Modelos base finales entrenados.


1371

14. Evaluacion final

In [15]:
test_meta = np.column_stack([
    final_xgb.predict_proba(X_test_final)[:, 1],
    final_lgbm.predict_proba(X_test_final)[:, 1],
    final_dnn.predict(X_test_final, verbose=0).reshape(-1)
]).astype("float32")

y_prob = meta_learner.predict_proba(test_meta)[:, 1]
y_pred = (y_prob >= 0.5).astype("int32")

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print("PCA activo:" if cfg.use_pca else "PCA inactivo:")
print("Accuracy :", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall   :", round(rec, 4))
print("F1-Score :", round(f1, 4))
print("\nMatriz de confusion:")
print(cm)
print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

PCA activo:
Accuracy : 0.8854
Precision: 0.9926
Recall   : 0.7634
F1-Score : 0.863

Matriz de confusion:
[[94692   486]
 [20220 65239]]

Reporte de clasificacion:
              precision    recall  f1-score   support

           0     0.8240    0.9949    0.9014     95178
           1     0.9926    0.7634    0.8630     85459

    accuracy                         0.8854    180637
   macro avg     0.9083    0.8791    0.8822    180637
weighted avg     0.9038    0.8854    0.8833    180637



15. Guardado de resultados

In [16]:
results = pd.DataFrame([{
    "use_pca": cfg.use_pca,
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1_score": f1
}])

results.to_csv(
    f"zero_day_results_pca_{cfg.use_pca}.csv",
    index=False
)

print("Resultados guardados.")

Resultados guardados.
